# 01 · First look at the Netflix Top 10 data

**Goal:** turn Netflix's weekly Top 10 file into one row per title-season, then label each one a **hit** (3+ weeks in the global Top 10) or not.

**How to use this notebook:** run each cell in order (Shift + Enter). Read the note above each cell first, and look at the output before moving on. Nothing here changes your raw data.

What you'll learn:
1. Loading a data file with **pandas** (the standard Python tool for tables)
2. Checking what's in it before trusting it
3. Rolling weekly rows up into one row per title (**groupby**)
4. Creating a label and checking its balance
5. Spotting a data trap: titles whose story isn't finished yet

## 1. Load the file

`pd.read_csv` reads a table from a file. Netflix's file uses tabs between columns instead of commas, so we pass `sep="\t"`.

The path starts with `../` because this notebook lives in the `notebooks/` folder, and the data lives one level up in `data/raw/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

weekly = pd.read_csv("../data/raw/all-weeks-global.tsv", sep="\t")
weekly.head()

## 2. What's in it?

Before analysing anything, check the size, the columns, and the date range. This catches problems early (a half-downloaded file, a wrong column name).

- `shape` = (rows, columns)
- Each row is **one title's position in one week's Top 10**, so a show that charted for 5 weeks appears 5 times.

In [ ]:
print("Rows, columns:", weekly.shape)
print("Weeks covered:", weekly["week"].min(), "to", weekly["week"].max(), f"({weekly['week'].nunique()} weeks)")
print()
print(weekly["category"].value_counts())

**Column guide**

| Column | Meaning |
|---|---|
| `week` | The Sunday that ends the week |
| `category` | Films or TV, English or Non-English: four separate Top 10 lists |
| `weekly_rank` | 1 to 10 |
| `show_title` / `season_title` | The show and, for TV, the season (`N/A` for films) |
| `weekly_hours_viewed` | Total hours watched that week |
| `weekly_views` | Hours viewed ÷ runtime: Netflix's main popularity number |
| `cumulative_weeks_in_top_10` | How many weeks it has been in the Top 10 so far |

**Try it:** pick one show you know and look at its rows. Change the title below and rerun.

In [ ]:
weekly[weekly["show_title"] == "Wednesday"]

## 3. One row per title-season

The model predicts one answer per title-season, so we need one row each. `groupby` collects all the weekly rows for the same title-season, and `agg` says how to summarise each column:

- **weeks_in_top10**: the highest `cumulative_weeks_in_top_10` it reached
- **first_week / last_week**: when it entered and when it was last seen
- **best_rank**: its highest position (1 is best)
- **peak_weekly_views**: its biggest single week

We use `season_title` as the key for TV (so Season 1 and Season 2 count separately) and `show_title` for films.

**Gotcha:** films have `N/A` as their season, and pandas reads the text "N/A" as *missing* (`NaN`). If we grouped on it as-is, `groupby` would silently drop every film. `fillna` swaps those blanks for the film's title. Silent data loss like this is one of the most common bugs in data work, which is why the next cell prints the count per category.

In [ ]:
weekly["title_key"] = weekly["season_title"].fillna(weekly["show_title"])

titles = (
    weekly.groupby(["category", "title_key"])
    .agg(
        show_title=("show_title", "first"),
        weeks_in_top10=("cumulative_weeks_in_top_10", "max"),
        first_week=("week", "min"),
        last_week=("week", "max"),
        best_rank=("weekly_rank", "min"),
        peak_weekly_views=("weekly_views", "max"),
    )
    .reset_index()
)
print(len(titles), "title-seasons")
print(titles["category"].value_counts())  # all four categories should appear
titles.sort_values("weeks_in_top10", ascending=False).head(10)

## 4. A data trap: stories that aren't finished

A title that entered the Top 10 last week has only 1 week **so far**, but it might go on to chart for 5. If we called it a "non-hit" now, the label would be wrong.

This is called **censoring**: the outcome hasn't happened yet. The fix is to leave out titles that entered too recently to have had a fair chance at 3 weeks. Here, that means anything that first charted in the last 2 weeks of the file and hasn't reached 3 weeks yet.

In [ ]:
latest_week = weekly["week"].max()
recent_weeks = sorted(weekly["week"].unique())[-2:]

still_open = titles["first_week"].isin(recent_weeks) & (titles["weeks_in_top10"] < 3)
print("Too recent to judge:", still_open.sum(), "title-seasons")
titles[still_open][["category", "title_key", "first_week", "weeks_in_top10"]]

In [ ]:
titles = titles[~still_open].copy()
print(len(titles), "title-seasons kept")

## 5. Create the label

`hit = 1` if a title-season spent 3 or more weeks in the global Top 10, otherwise `0`.

Then check the **balance**: what share are hits? If hits were only 2% of titles, the model could score 98% accuracy by always saying "no." That's the baseline trap we discussed.

In [ ]:
titles["hit"] = (titles["weeks_in_top10"] >= 3).astype(int)

print(f"Overall hit rate: {titles['hit'].mean():.0%}")
titles.groupby("category")["hit"].agg(titles="count", hit_rate="mean").style.format({"hit_rate": "{:.0%}"})

## 6. Picture it

How many weeks do titles usually last? The dashed line marks the 3-week cutoff. Everything to its right counts as a hit.

In [ ]:
counts = titles["weeks_in_top10"].clip(upper=15).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.index, counts.values, color="#4C78A8")
ax.axvline(2.5, color="black", linestyle="--", linewidth=1)
ax.text(2.7, counts.max() * 0.9, "hit = 3+ weeks", fontsize=10)
ax.set_xlabel("Weeks in global Top 10 (15 = 15 or more)")
ax.set_ylabel("Number of title-seasons")
ax.set_title("Most titles drop out within 2 weeks")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 7. Save the result

This table is the start of your training data. Saving it to `data/processed/` keeps the raw file untouched and gives the next notebook something to build on.

In [ ]:
titles.to_csv("../data/processed/top10_titles.csv", index=False)
print("Saved", len(titles), "rows to data/processed/top10_titles.csv")

## Your turn (optional)

Short questions to answer in a new cell. Each one is a few lines of pandas.

1. Which category has the highest hit rate, and why might that be?
2. Do titles that reach **rank 1** almost always become hits? (Hint: group by `best_rank == 1`.)
3. How has the hit rate changed by year? (Hint: `titles["first_week"].str[:4]` gives the year.)

**Remember the gap:** this file only includes titles that made the Top 10 at least once. Originals that never charted are missing, and they're your clearest non-hits. The "What We Watched" reports fill that gap next.